In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [4]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([8])


In [6]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


In [7]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


In [8]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [9]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.2824,  0.1743,  0.2522,  0.6155, -0.2754,  0.6193,  0.0141, -0.5678,
          0.2202,  0.1042, -0.0078,  0.2007,  0.0789, -0.3036, -0.1604,  0.5037,
          0.0882, -0.3052, -0.1535,  0.3731],
        [-0.1150,  0.3174,  0.0601,  0.5341, -0.3094,  0.6565, -0.1329, -0.4613,
          0.2061,  0.0611,  0.1169, -0.1010,  0.1711, -0.3793,  0.2396,  0.3961,
          0.1674, -0.0668,  0.3040,  0.4227],
        [-0.1406,  0.0545,  0.1176,  0.3219, -0.3792,  0.6286,  0.1923, -0.4249,
          0.0450, -0.0822, -0.0914, -0.2622,  0.0795, -0.4373, -0.1267,  0.3293,
          0.0792, -0.0287, -0.2515,  0.3204]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.1743, 0.2522, 0.6155, 0.0000, 0.6193, 0.0141, 0.0000, 0.2202,
         0.1042, 0.0000, 0.2007, 0.0789, 0.0000, 0.0000, 0.5037, 0.0882, 0.0000,
         0.0000, 0.3731],
        [0.0000, 0.3174, 0.0601, 0.5341, 0.0000, 0.6565, 0.0000, 0.0000, 0.2061,
         0.0611, 0.1169, 0.0000, 0.1711, 0.0000, 0.23

In [10]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

In [11]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

In [12]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0023,  0.0344, -0.0135,  ..., -0.0321,  0.0278,  0.0320],
        [ 0.0136,  0.0085,  0.0203,  ..., -0.0265,  0.0011,  0.0258]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([-0.0293, -0.0174], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[-0.0430,  0.0263,  0.0209,  ..., -0.0307,  0.0007, -0.0158],
        [-0.0403,  0.0367,  0.0158,  ..., -0.0197,  0.0303, -0.0406]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | 